# Week 14 과제 · AI 디버깅 · 테스트 · 리팩토링

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week14/assignment.ipynb)

## 제출 방법

1. **파일 → 드라이브에 사본 저장** 후 파일 이름을 `week14_학번_이름.ipynb`로 바꿉니다.
2. `# TODO` 부분을 채우고 각 문제 아래 **자동 확인 셀**이 ✅인지 확인합니다.
3. 버그 보고서, AI 활용 기록, 성찰을 작성합니다.
4. **런타임 → 다시 시작 및 모두 실행** 후 `.ipynb` 파일을 제출합니다.

| 문제 | 내용 | 배점 |
|---|---|---:|
| 1 | 버그 3개 찾아 고치기 + 버그 보고서 | 2 |
| 2 | 버그를 잡는 테스트 만들기 | 3 |
| 3 | 결과는 그대로, 코드는 깔끔하게 (리팩토링) | 3 |
| 4 | AI와 함께 pandas 버그 고치기 | 2 |
| | **합계** | **10** |

## 준비: 데이터와 자동 확인 도우미

In [ ]:
import inspect
import numpy as np
import pandas as pd
from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

CAFE = pd.read_csv(get_data("cafe_sales.csv"), parse_dates=["date"])
CAFE_BACKUP = CAFE.copy()


def check(desc, test):
    """자동 확인 도우미: test는 True/False를 돌려주는 함수(lambda)입니다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")

---
## 문제 1. 버그 3개 찾아 고치기 + 버그 보고서 (2점)

`score_stats(scores)`는 점수 리스트(결측은 `None`)를 받아 딕셔너리를 돌려줘야 합니다.

| 키 | 의미 |
|---|---|
| `"n"` | 결측을 뺀 점수 개수 |
| `"mean"` | 결측을 뺀 평균 (소수 첫째 자리), 점수가 없으면 `None` |
| `"pass"` | 60점 **이상**인 개수 |
| `"top"` | 상위 3개 점수, 큰 순서 (3개 미만이면 있는 만큼) |

아래 코드에는 **버그가 3개** 있습니다. 작은 입력으로 재현하고, 원인을 찾아 고치세요. (코드를 처음부터 새로 쓰지 말고 **최소한으로** 고칩니다.) 그리고 버그 보고서를 작성합니다.

In [ ]:
def score_stats(scores):
    valid = [s for s in scores if s]
    n = len(valid)
    mean = round(sum(valid) / n, 1) if n else None
    passed = sum(1 for s in valid if s > 60)
    top = sorted(valid)[:3]
    return {"n": n, "mean": mean, "pass": passed, "top": top}

In [ ]:
check("기본", lambda: score_stats([70, 85, 92, 55]) == {"n": 4, "mean": 75.5, "pass": 3, "top": [92, 85, 70]})
check("0점도 점수 (결측 아님)", lambda: score_stats([0, 100, None]) == {"n": 2, "mean": 50.0, "pass": 1, "top": [100, 0]})
check("60점은 합격", lambda: score_stats([60, 59.9])["pass"] == 1)
check("빈 입력", lambda: score_stats([]) == {"n": 0, "mean": None, "pass": 0, "top": []})
check("모두 결측", lambda: score_stats([None, None])["mean"] is None)

### 버그 보고서

| 번호 | 재현 입력 | 기대 결과 | 실제 결과 | 원인 | 수정 |
|---|---|---|---|---|---|
| 1 |  |  |  |  |  |
| 2 |  |  |  |  |  |
| 3 |  |  |  |  |  |

---
## 문제 2. 버그를 잡는 테스트 만들기 (3점)

`clean_price(text)`의 명세는 다음과 같습니다.

| 입력 예 | 결과 | 규칙 |
|---|---|---|
| `"2,500"` | `2500` | 쉼표 제거 후 정수 |
| `" ₩3,200 "` | `3200` | 앞뒤 공백, `₩` 제거 |
| `"4500원"` | `4500` | 끝의 `원` 제거 |
| `"0"` | `0` | 0원은 유효 |
| `""`, `"무료"`, `"1.5"` | `None` | 정수로 바꿀 수 없으면 `None` |
| `"-500"` | `None` | 음수는 `None` |

`my_tests(fn)`은 `fn`이 명세를 **모두 지키면 `True`**, 하나라도 어기면 `False`를 돌려줍니다. `cases`에 **(입력, 기대 결과)를 6개 이상** 채우세요.
여러분의 테스트는 **정답 함수는 통과**시키고, 아래 **버그 함수 3개는 모두 잡아내야** 합니다.

In [ ]:
def clean_price_ok(text):                 # 정답 (수정하지 마세요)
    t = text.strip().replace("₩", "").replace(",", "")
    if t.endswith("원"):
        t = t[:-1]
    try:
        value = int(t)
    except ValueError:
        return None
    return value if value >= 0 else None


def bug_negative(text):                   # 버그 함수 1: 음수를 그대로 돌려준다
    t = text.strip().replace("₩", "").replace(",", "").removesuffix("원")
    try:
        return int(t)
    except ValueError:
        return None


def bug_zero(text):                       # 버그 함수 2: 0을 None으로 돌려준다
    value = clean_price_ok(text)
    return value or None


def bug_won(text):                        # 버그 함수 3: '원'을 처리하지 않는다
    t = text.strip().replace("₩", "").replace(",", "")
    try:
        value = int(t)
    except ValueError:
        return None
    return value if value >= 0 else None


def my_tests(fn):
    cases = [
        # (입력, 기대 결과)
        # TODO: 6개 이상 작성하세요. 예: ("2,500", 2500)
    ]
    if len(cases) < 6:
        return False
    for text, expected in cases:
        try:
            if fn(text) != expected:
                return False
        except Exception:
            return False
    return True

In [ ]:
check("내 테스트 6개 이상 + 정답 통과", lambda: my_tests(clean_price_ok) is True)
check("버그 함수 1 (음수) 잡음", lambda: my_tests(bug_negative) is False)
check("버그 함수 2 (0원) 잡음", lambda: my_tests(bug_zero) is False)
check("버그 함수 3 ('원') 잡음", lambda: my_tests(bug_won) is False)

---
## 문제 3. 결과는 그대로, 코드는 깔끔하게 (3점)

아래 `payment_report_old`는 **동작은 맞지만** 읽기 어렵습니다. 같은 결과를 내는 `payment_report(cafe)`로 리팩토링하세요.

**리팩토링 조건** (자동 확인이 검사합니다)

1. 결과가 `payment_report_old`와 **정확히 같아야** 합니다 (원본 데이터, 일부 데이터 모두).
2. 결제 수단 하나를 요약하는 도우미 함수 `summarize_payment(rows)`를 만들고 `payment_report`에서 사용합니다.
3. 결제 수단 목록은 상수 `PAYMENTS`로 둡니다.
4. `payment_report`는 **10줄 이하**, 입력 DataFrame을 바꾸지 않습니다.

In [ ]:
def payment_report_old(cafe):
    cafe = cafe.copy()
    out = {}
    a = cafe[cafe["payment"] == "카드"]
    out["카드"] = {"건수": len(a), "매출": int((a["unit_price"] * a["qty"]).sum()),
                 "비중": round(len(a) / len(cafe) * 100, 1) if len(cafe) else 0.0}
    b = cafe[cafe["payment"] == "모바일"]
    out["모바일"] = {"건수": len(b), "매출": int((b["unit_price"] * b["qty"]).sum()),
                  "비중": round(len(b) / len(cafe) * 100, 1) if len(cafe) else 0.0}
    c = cafe[cafe["payment"] == "현금"]
    out["현금"] = {"건수": len(c), "매출": int((c["unit_price"] * c["qty"]).sum()),
                 "비중": round(len(c) / len(cafe) * 100, 1) if len(cafe) else 0.0}
    return out

In [ ]:
PAYMENTS = []          # TODO


def summarize_payment(rows, total_orders):
    raise NotImplementedError("코드를 작성하세요")


def payment_report(cafe):
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
check("전체 데이터 결과 동일", lambda: payment_report(CAFE) == payment_report_old(CAFE))
check("일부 데이터(3월) 결과 동일", lambda: payment_report(CAFE[CAFE["date"].dt.month == 3]) == payment_report_old(CAFE[CAFE["date"].dt.month == 3]))
check("빈 데이터 결과 동일", lambda: payment_report(CAFE.iloc[0:0]) == payment_report_old(CAFE.iloc[0:0]))
check("PAYMENTS 상수", lambda: PAYMENTS == ["카드", "모바일", "현금"])
check("도우미 함수를 사용", lambda: "summarize_payment" in inspect.getsource(payment_report))
check("payment_report 10줄 이하", lambda: len(inspect.getsource(payment_report).strip().splitlines()) <= 10)
check("입력 불변", lambda: (payment_report(CAFE), CAFE.equals(CAFE_BACKUP))[1])

---
## 문제 4. AI와 함께 pandas 버그 고치기 (2점)

`daily_ice_ratio(cafe)`는 날짜별 **아이스 음료 비율(%)** 을 Series로 돌려줘야 합니다.

- 비율 = ICE **판매 수량** ÷ (ICE + HOT **판매 수량**) × 100, 소수 첫째 자리
- 디저트(`temp_type == "-"`)는 계산에서 제외
- 입력 DataFrame은 바꾸지 않음

아래 코드는 실행은 되지만 결과가 틀립니다. **AI에게 좋은 질문(최소 재현 + 기대/실제)** 을 해서 원인을 찾고 고치세요.

In [ ]:
def daily_ice_ratio(cafe):
    drinks = cafe[cafe["temp_type"] != "-"]
    drinks["is_ice"] = drinks["temp_type"] == "ICE"
    ratio = drinks.groupby("date")["is_ice"].mean() * 100
    return ratio.round(1)

In [ ]:
tiny = pd.DataFrame({"date": pd.to_datetime(["2025-03-03"] * 4 + ["2025-03-04"] * 2),
                     "temp_type": ["ICE", "HOT", "HOT", "-", "ICE", "ICE"],
                     "qty": [3, 1, 1, 5, 1, 2]})
tiny_before = tiny.copy()
check("수량 기준 (3/5 = 60%)", lambda: daily_ice_ratio(tiny).loc["2025-03-03"] == 60.0)
check("모두 ICE면 100%", lambda: daily_ice_ratio(tiny).loc["2025-03-04"] == 100.0)
check("입력 불변 (새 열 없음)", lambda: (daily_ice_ratio(tiny), tiny.equals(tiny_before))[1])
check("전체 데이터 첫날", lambda: daily_ice_ratio(CAFE).iloc[0] == 12.0)

### AI 활용 기록 (문제 4)

| 항목 | 내용 |
|---|---|
| 사용한 AI 도구 |  |
| AI에게 보낸 질문 (최소 재현 코드, 기대·실제 결과 포함) |  |
| AI가 설명한 원인 |  |
| AI 수정안을 그대로 썼나? 다른 데이터로 검증한 방법은? |  |

---
## 성찰 (필수)

1. 오류 메시지가 없는 "조용한 버그"를 찾을 때 가장 도움이 된 방법은?
2. 테스트가 있었기 때문에 리팩토링이 안전했던 순간이 있었나요?
3. AI의 수정이 테스트만 통과시키는 가짜 수정인지 어떻게 확인할 수 있을까요?

(여기에 작성)